# 08. LangGraph Testing — state와 checkpoint를 검증하기

## 학습 목표

- LangGraph node와 compiled graph를 분리해 테스트합니다.
- state assertion으로 회귀를 잡습니다.
- checkpointer가 필요한 테스트와 필요 없는 테스트를 구분합니다.

In [ ]:
from dotenv import load_dotenv
import os

load_dotenv(override=True)

In [ ]:
# Observability 설정 — 키가 없으면 비활성 상태로 둡니다.
if os.environ.get("LANGSMITH_TRACING", "").lower() == "true":
    os.environ.setdefault("LANGSMITH_PROJECT", "agent-notebooks")
lf_config = {}

In [ ]:
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import InMemorySaver

class CounterState(TypedDict):
    count: int

## 8.1 node unit test

node는 순수 함수처럼 입력 state와 출력 update를 검증합니다.

In [ ]:
def increment(state: CounterState) -> dict:
    return {"count": state["count"] + 1}

assert increment({"count": 1}) == {"count": 2}
print("node unit test passed")

## 8.2 graph integration test

compile된 graph는 end-to-end state를 검증합니다.

In [ ]:
builder = StateGraph(CounterState)
builder.add_node("increment", increment)
builder.add_edge(START, "increment")
builder.add_edge("increment", END)
graph = builder.compile()

assert graph.invoke({"count": 0})["count"] == 1

## 8.3 checkpoint smoke

thread_id가 필요한 실행은 config까지 테스트해야 합니다.

In [ ]:
checkpointer = InMemorySaver()
graph_with_memory = builder.compile(checkpointer=checkpointer)
config = {"configurable": {"thread_id": "test-1"}}

result = graph_with_memory.invoke({"count": 2}, config=config)
assert result["count"] == 3

---

## 정리

| 항목 | 내용 |
|---|---|
| **다룬 기술** | node unit test, graph integration test, checkpoint smoke |
| **핵심 개념** | LangGraph 테스트는 state update와 compiled graph behavior를 따로 검증합니다. |

**참고 문서:**
- `docs/langgraph/test.md`
- `docs/langgraph/checkpointers.md`
- `docs/langchain/test/unit-testing.md`